In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
import json
import torch
from transformers import AutoTokenizer, AutoModel


class DNABERT2EmbeddingExtractor:
    """
    Класс для извлечения эмбеддингов из последовательностей ДНК/РНК
    с помощью модели DNABERT-2.
    Метод extract_embeddings принимает список строк (последовательностей)
    и возвращает numpy-массив эмбеддингов.
    """
    def __init__(self, model_name: str = "zhihan1996/DNABERT-2-117M", device: int = 0):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True, return_dict=True)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.model.eval()

    def extract_embeddings(self, sequences, batch_size: int = 16) -> np.ndarray:
        all_embeds = []
        for i in tqdm(range(0, len(sequences), batch_size), desc="Extracting embeddings"):
            batch = sequences[i : i + batch_size]
            enc = self.tokenizer(batch,
                                  padding=True,
                                  truncation=True,
                                  return_tensors="pt")
            enc = {k: v.to(self.device) for k, v in enc.items()}
            with torch.no_grad():
                out = self.model(**enc)
            cls_embeds = out[0][:, 0, :].cpu().numpy()
            all_embeds.append(cls_embeds)
        return np.vstack(all_embeds)


ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", trust_remote_code=True)

train_ds, test_ds = ds['train'], ds['test']

extractor = DNABERT2EmbeddingExtractor()

PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
BATCH_SIZE = 16
PATH_TO_SAVE_OUTPUTS = '.'

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
    seqs_te, y_te = te['sequence'], np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnabert2_task-{task}_baseline.json', 'w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, extractor, ks=(1, 5, 10, 20), trials=5):
    res = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr, y_tr = tr['sequence'], np.array(tr['label'])
        seqs_te, y_te = te['sequence'], np.array(te['label'])

        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        res[task] = {}
        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]
                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf, y_k)
                p = clf.predict(Xt)
                accs.append(accuracy_score(y_te, p))
                f1s.append(f1_score(y_te, p, average='macro'))
            res[task][k] = {
                'accuracy': float(np.mean(accs)),
                'f1_score': float(np.mean(f1s))
            }
            with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnabert2_task-{task}_k-{k}.json', 'w') as f:
                json.dump(res, f, indent=4)
    return res

results_kshot = few_shot(train_ds, test_ds, extractor)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE_OUTPUTS}/results_dnabert2.json', 'w') as f:
    json.dump(output, f, indent=4)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'InstaDeepAI/nucleotide_transformer_downstream_tasks' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Few-shot: 100%|██████████| 18/18 [05:04<00:00, 16.92s/it]


In [5]:
from pathlib import Path

Path('InstaDeepAI/nucleotide-transformer-v2-50m-multi-species').stem

'nucleotide-transformer-v2-50m-multi-species'